# ML 00 — Literature review, problem framing and evaluation plan

## Purpose of this notebook

This notebook frames **Phase 1 (Machine Learning)** of the project *Microclimate Modelling via Neural Surrogates and Computational Fluid Dynamics*. It documents the problem, the current dataset-supported question, the theory behind the chosen ML methods, and the evidence required for a defensible evaluation.

> **Scope discipline:** Phase 1 evaluates a scalar AML baseline. The project brief's later raster-to-field CNN/U-Net is a Phase 2 direction; it is not claimed to be implemented by this notebook set.

## 1. Problem statement and research questions

**Project question.** How well can layout-level geometric summaries and wind direction predict the canopy-averaged wind-speed scalar for idealized UrbanTALES cases, especially when an entire plan family is held out?

**Target.** `target_U_canopy_mean_m_s`, joined from the `U` column in `UrbanTALES-flowdata.csv`. The supplied project documentation describes `U` as a canopy-averaged wind-speed quantity in m/s. The target is one scalar per simulation case, not a spatial velocity map.

**Unit of observation.** One layout under one wind direction. In this subset there are 224 simulation cases, 112 unique layouts and 16 family groups. Two directions for the same layout are related observations, so they must not be treated as fully independent random rows.

**Research questions**
1. How are target values and geometry features distributed, and what data-quality limitations matter?
2. Do geometry summaries and wind-direction features contain useful predictive signal beyond a training-fold mean baseline?
3. Do nonlinear models improve over a regularized linear model under family-held-out evaluation?
4. How much do scores change when the split is made by family versus by layout?
5. Which claims are supported by scalar targets, and which spatial-flow claims remain out of scope?

## 2. Literature context and how it informs Phase 1

The faculty brief suggests the following pre-reads. A complementary peer-reviewed review by Brunton, Noack and Koumoutsakos surveys machine learning for fluid mechanics and discusses both opportunities and limitations of data-driven flow modelling. These readings motivate the broader flow-surrogate context; none is evidence that the current scalar baseline itself is a neural or spatial-field surrogate.

- **Goodfellow et al. (2014), “Generative Adversarial Nets.”** Introduces adversarial generative modelling. It is relevant as background on learning distributions, but a GAN is not required for this Phase 1 regression task and is not used in the current pipeline.
- **Farimani et al. (2017), “Deep Learning for Wall Bounded Turbulent Flows.”** Listed as a suggested pre-read in the project brief. It motivates data-driven modelling of flow behaviour. The precise bibliographic metadata should be cross-checked against the course-provided reference before the final IEEE report; this notebook does not invent a venue or DOI.
- **Brunton, Noack and Koumoutsakos (2020), “Machine Learning for Fluid Mechanics.”** *Annual Review of Fluid Mechanics*, 52, 477–508, DOI: 10.1146/annurev-fluid-010719-060214. This review places regression, reduced-order modelling and other ML methods in the broader fluid-mechanics context and emphasizes that model choice, physical knowledge and generalization must be considered together.

**Implication for the current phase:** begin with transparent scalar regression baselines and a defensible validation protocol. The brief explicitly describes Random Forest/XGBoost on geometric summaries as the AML baseline. Spatially structured CNN/U-Net learning belongs to the later DL phase and requires paired spatial fields, aligned grids and field-level metrics.

## 3. Why these Phase 1 models?

| Method | Theoretical role | What it tests | Important limitation |
|---|---|---|---|
| Training-fold mean baseline | No-feature reference | Whether features add value at all | Cannot represent layout or direction effects |
| Ridge regression | Linear model with L2 penalty | Whether mostly linear relationships are adequate; shrinkage controls coefficient magnitude | Limited nonlinear interaction capacity |
| Random Forest | Bagged decision trees with feature subsampling | Nonlinear thresholds and feature interactions | Extrapolation beyond observed feature ranges is weak; impurity importance can be biased |
| XGBoost | Sequential gradient-boosted trees | Whether boosting improves predictive error on tabular features | Sensitive to hyperparameters; importance is not causal evidence |

**Why regularization matters.** Ridge minimizes squared error plus an L2 penalty, `RSS + α Σ β_j²`. The penalty discourages unstable large coefficients, especially when predictors are correlated. Feature scaling is fitted within each training fold so test data does not influence the learned scaling parameters.

**Why group validation matters.** Rows share geometry and family structure. If sibling cases occur in both training and test sets, performance can look better than performance on an unseen family. The primary estimate therefore uses family-held-out folds; layout-held-out and family leave-one-group-out protocols are secondary sensitivity checks.

## 4. Metrics and interpretation

For observed values \(y_i\) and predictions \(\hat y_i\):

- **MAE:** \(\frac{1}{n}\sum_i |y_i-\hat y_i|\). Typical absolute error in m/s; easy to interpret.
- **RMSE:** \(\sqrt{\frac{1}{n}\sum_i (y_i-\hat y_i)^2}\). Penalizes larger errors more strongly.
- **Normalized RMSE:** RMSE divided by the observed target range in this dataset. It is scale-relative but depends on the observed range, so the denominator must be reported.
- **R²:** \(1-\frac{\sum_i(y_i-\hat y_i)^2}{\sum_i(y_i-\bar y)^2}\). May be negative when predictions are worse than predicting the test-set mean; it is not an accuracy percentage.

Report all metrics on out-of-fold predictions and compare against the mean baseline. Uncertainty intervals are approximate and must respect the family clustering; they do not remove uncertainty caused by having only 16 families.

## 5. Feature-engineering logic and leakage policy

- Plan-area density is a compact proxy for built-up fraction and can have nonlinear effects, so its square is tested as a simple nonlinear term.
- Height mean/dispersion/range describe vertical heterogeneity; the coefficient of variation (standard deviation divided by mean) makes dispersion relative to the typical height.
- Wind direction is circular. Sine/cosine encoding avoids treating 0° and 360° as far apart. The current data contains 0° and 45° only, so conclusions about all possible directions are not justified.
- Configuration and density classes are categorical and are one-hot encoded inside the model pipeline.
- `simulation_id`, `layout_id`, `family_id`, and the target are identifiers/grouping fields, not predictors. The original metadata label is retained for traceability but excluded because it may duplicate a folder-derived configuration label.
- Any statistic learned from data (imputation, scaling, encoding) is fitted on the training fold only.

## 6. Reproducibility and evidence checklist

The notebooks should be run in order: ML_01 → ML_02 → ML_03 → ML_04 → ML_05 → ML_06 → ML_07. ML_01 regenerates `data/processed/layout_level_dataset.csv` and the matching data dictionary from raw files. ML_03 generates the engineered table. Model notebooks calculate predictions/metrics in the current run and display plots inline; the `results/` directory is an optional runtime output, not a source of model inputs.

Before claiming a result in the report or viva, be ready to explain: (a) the raw-to-target join, (b) the target's physical meaning and units, (c) why family-held-out validation is primary, (d) how every feature is computed, (e) why each metric is reported, (f) what the models cannot predict, and (g) why spatial-field/temperature claims are out of scope for this Phase 1 baseline.

In [ ]:
from pathlib import Path
import pandas as pd

_candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _candidates if (p / "data" / "raw").exists() and (p / "Machine Learning Pipeline").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the project repository.")
raw = ROOT / "data" / "raw"
processed = ROOT / "data" / "processed"
print("Repository root:", ROOT)
print("Raw inputs present:", all((raw / f).exists() for f in ["metadata_ideal.json", "metadata.csv", "UrbanTALES-flowdata.csv"]))
print("Processed dataset currently present:", (processed / "layout_level_dataset.csv").exists())
print("Next: run ML_01 to regenerate processed data, then continue in numerical order.")

## References

1. I. Goodfellow et al., “Generative Adversarial Nets,” *Advances in Neural Information Processing Systems*, vol. 27, 2014. (Suggested pre-read in the project brief.)
2. A. Farimani et al., “Deep Learning for Wall Bounded Turbulent Flows,” 2017. (Title/year as supplied in the project brief; verify complete publication metadata against the faculty source before submitting the IEEE report.)
3. S. L. Brunton, B. R. Noack, and P. Koumoutsakos, “Machine Learning for Fluid Mechanics,” *Annual Review of Fluid Mechanics*, vol. 52, pp. 477–508, 2020, doi: 10.1146/annurev-fluid-010719-060214.
4. UrbanTALES dataset documentation and datasheet supplied with this project under `data/raw/`; cite the dataset publication and licence details using the exact metadata from the provided datasheet in the final report.